In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
# response=requests.get("https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=471")

In [3]:
# response.json()['results']

In [4]:
# df=pd.DataFrame(response.json()['results'])
# df.head(1)

In [5]:
# df.drop(columns=['backdrop_path','genre_ids','poster_path','video'],inplace=True)


In [6]:
df=pd.DataFrame()
for i in range(1,5):
  response=requests.get(f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={i}")
  temp_df=pd.DataFrame(response.json()['results'])
  temp_df.drop(columns=['backdrop_path','genre_ids','poster_path','video'],inplace=True)
  df=pd.concat([df,temp_df],ignore_index=True)
  print(f"page{i}")


page1
page2
page3
page4


In [7]:
df.shape

(80, 11)

In [8]:
df.head(1)

,adult,id,title,original_language,original_title,overview,popularity,release_date,softcore,vote_average,vote_count
0,False,1560520,Batman: Knightfall Part 1: Knightfall,en,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...",83.7608,2026-06-23,False,9.175,312


## Removing html tags in text

In [9]:
import re

url_pattern = r'https?://\S+|www\.\S+'

df['overview'].str.contains(url_pattern, regex=True, na=False).sum()          # NO LINKS IN OVERVIEW FEATURE

np.int64(0)

In [10]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)
df['overview']=df['overview'].apply(remove_html_tags)

## Lowercasing the fetures text

In [11]:
df['overview']=df['overview'].str.lower()


In [12]:
df['overview'].head()

,overview
0,"arkham asylum has been destroyed, and all its ..."
1,"avatar aang, the world's last airbender, learn..."
2,"a small woodland creature and a majestic bird,..."
3,two women discover they were both scammed by t...
4,two mexican officers must survive the final ho...


## Removing Punctuation


In [13]:
import string

exclude = string.punctuation


def count_punct(text):
    count = 0

    for char in text:
        if char in exclude:
            count += 1

    return count
df['overview'].apply(count_punct).sum()

np.int64(516)

In [14]:
# import string
# import re

# exclude = string.punctuation

# pattern = '[' + re.escape(exclude) + ']'

# df['overview'].str.contains(pattern, regex=True, na=False).sum()

In [15]:
import re
import string

exclude = string.punctuation

df['overview'] = df['overview'].str.replace(
    '[' + re.escape(exclude) + ']',
    '',
    regex=True
)

In [16]:
df['overview'].apply(count_punct).sum()

np.int64(0)

## Spelling correction

In [17]:
df['overview'][1]

'avatar aang the worlds last airbender learns of an ancient power that could save his culture from extinction with the help of his friends he embarks on a global quest to find it before it falls into the wrong hands and threatens to upend the peace they sacrificed everything to achieve'

In [18]:
!pip install textblob

In [19]:
from tqdm import tqdm
from textblob import TextBlob

tqdm.pandas()

df['corrected_overview'] = df['overview'].progress_apply(
    lambda text: str(TextBlob(str(text)).correct())
)

100%|██████████| 80/80 [01:09<00:00,  1.15it/s]


In [20]:
df['overview']=df['corrected_overview']

In [21]:
df['overview'][0]

'abraham asylum has been destroyed and all its inmates have been unleashed upon gothic city as batman races to round up some of his greatest enemies he is pushed to his physical and mental limits and into a final confrontation with a new threat the man called bane'

## Emoji removal or Demojize

In [22]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.3 MB/s eta 0:00:00


In [23]:
import emoji

def has_emoji(text):
    if not isinstance(text, str):
        return False
    return emoji.emoji_count(text) > 0

# Create a mask for rows containing emojis
emoji_mask = df['overview'].apply(has_emoji)

# Display the count and the first few rows found
print(f"Number of rows with emojis: {emoji_mask.sum()}")

if emoji_mask.any():
    display(df[emoji_mask][['title', 'overview']].head())
else:
    print("No emojis were detected in the overview column.")

Number of rows with emojis: 0
No emojis were detected in the overview column.


In [24]:
import re
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [25]:
df['overview']=df['overview'].apply(remove_emoji)

In [26]:
# emoji.demojize()

## Stop words removal

In [27]:
from nltk.corpus import stopwords

In [28]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [29]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    new_text = []

    for word in text.split():
        if word not in stop_words:
            new_text.append(word)

    return " ".join(new_text)

df['overview'] = df['overview'].apply(remove_stopwords)

In [30]:
df['overview'][0]

'abraham asylum destroyed inmates unleashed upon gothic city batman races round greatest enemies pushed physical mental limits final confrontation new threat man called bane'

## Chatwords Conversion

In [31]:
chat_words = {

    # Common texting
    "U": "you",
    "UR": "your",
    "R": "are",
    "Y": "why",
    "YA": "you",
    "YEAH": "yes",
    "YES": "yes",
    "NO": "no",
    "PLS": "please",
    "PLZ": "please",
    "PPL": "people",
    "SOME1": "someone",
    "NE1": "anyone",
    "NVM": "never mind",
    "NM": "never mind",
    "NP": "no problem",
    "NBD": "no big deal",
    "IDK": "I do not know",
    "IDC": "I do not care",
    "IDGAF": "I do not care",
    "IK": "I know",
    "IKR": "I know right",
    "IMO": "in my opinion",
    "IMHO": "in my humble opinion",
    "TBH": "to be honest",
    "TBF": "to be fair",
    "FYI": "for your information",
    "BTW": "by the way",
    "ASAP": "as soon as possible",
    "ATM": "at the moment",
    "AFAIK": "as far as I know",
    "AFAIR": "as far as I remember",
    "B4": "before",
    "BC": "because",
    "CU": "see you",
    "CYA": "see you",
    "L8R": "later",
    "GR8": "great",
    "2DAY": "today",
    "2MORO": "tomorrow",
    "2NITE": "tonight",
    "4": "for",
    "2": "to",
    "W8": "wait",
    "WAT": "what",
    "PLS": "please",
    "THX": "thanks",
    "THNX": "thanks",
    "TY": "thank you",
    "TYSM": "thank you so much",
    "YW": "you are welcome",
    "WB": "welcome back",

    # Greetings
    "HI": "hello",
    "HEY": "hello",
    "HLO": "hello",
    "HII": "hello",
    "HIII": "hello",
    "GM": "good morning",
    "GMRNG": "good morning",
    "GN": "good night",
    "GNGT": "good night",
    "GE": "good evening",
    "GA": "good afternoon",

    # Laughing / reactions
    "LOL": "laughing out loud",
    "LMAO": "laughing my ass off",
    "LMFAO": "laughing my freaking ass off",
    "ROFL": "rolling on the floor laughing",
    "ROFLMAO": "rolling on the floor laughing",
    "HAHA": "laughing",
    "HAHAHA": "laughing",
    "HEHE": "laughing",
    "XD": "laughing",
    "XDXD": "laughing",
    "OMG": "oh my god",
    "OMFG": "oh my freaking god",
    "WTF": "what the heck",
    "WTH": "what the heck",
    "SMH": "shaking my head",
    "BRUH": "bro",
    "BRO": "brother",
    "FR": "for real",
    "FRFR": "for real for real",
    "ISTG": "I swear to god",
    "ONG": "on god",

    # Emotions / feelings
    "ILY": "I love you",
    "ILU": "I love you",
    "ILYSB": "I love you so much",
    "LYSM": "love you so much",
    "MISSU": "miss you",
    "MUCH": "much",
    "HBD": "happy birthday",
    "HB": "happy birthday",
    "HNY": "happy new year",
    "GL": "good luck",
    "GLHF": "good luck have fun",
    "GJ": "good job",
    "GG": "good game",
    "WP": "well played",
    "HF": "have fun",
    "GMTA": "great minds think alike",

    # Agreement / disagreement
    "AGREE": "agree",
    "DISAGREE": "disagree",
    "YUP": "yes",
    "YEP": "yes",
    "YEA": "yes",
    "NAH": "no",
    "NOPE": "no",
    "SURE": "sure",
    "KK": "okay",
    "K": "okay",
    "OK": "okay",
    "OKAY": "okay",
    "OKEY": "okay",
    "ALR": "alright",
    "ALRIGHT": "alright",

    # Conversation
    "WBU": "what about you",
    "HBU": "how about you",
    "HRU": "how are you",
    "HOWRU": "how are you",
    "WYD": "what are you doing",
    "WYA": "where are you",
    "WYM": "what you mean",
    "WDYM": "what do you mean",
    "WYWH": "wish you were here",
    "HMU": "hit me up",
    "DM": "direct message",
    "PM": "private message",
    "MSG": "message",
    "TXT": "text",
    "TC": "take care",
    "TTYL": "talk to you later",
    "TTYS": "talk to you soon",
    "TTFN": "ta-ta for now",
    "BBS": "be back soon",
    "BBL": "be back later",
    "BRB": "be right back",
    "BRT": "be right there",
    "GTG": "got to go",
    "G2G": "got to go",
    "G2BU": "good to be you",
    "AFK": "away from keyboard",

    # Social media
    "IG": "Instagram",
    "FB": "Facebook",
    "YT": "YouTube",
    "LI": "LinkedIn",
    "TIKTOK": "TikTok",
    "DM": "direct message",
    "PM": "private message",
    "RT": "retweet",
    "TL": "timeline",
    "OP": "original poster",
    "AMA": "ask me anything",
    "ICYMI": "in case you missed it",
    "TBT": "throwback Thursday",
    "OOTD": "outfit of the day",
    "FOMO": "fear of missing out",
    "JOMO": "joy of missing out",
    "IRL": "in real life",

    # Internet slang
    "GOAT": "greatest of all time",
    "GOATED": "greatest of all time",
    "BAE": "before anyone else",
    "BFF": "best friend forever",
    "BFFL": "best friends for life",
    "BESTIE": "best friend",
    "CRINGE": "embarrassing",
    "SUS": "suspicious",
    "SUSSY": "suspicious",
    "CAP": "lie",
    "NO CAP": "no lie",
    "BET": "okay",
    "W": "win",
    "L": "loss",
    "WTF": "what the heck",
    "LIT": "exciting",
    "FIRE": "excellent",
    "SLAY": "do very well",
    "BASED": "confidently authentic",
    "MID": "average",
    "RATIO": "receive more positive reactions",
    "VALID": "reasonable",
    "BASED": "reasonable",
    "LOWKEY": "quietly",
    "HIGHKEY": "obviously",
    "VIBE": "feeling",
    "VIBES": "feelings",
    "GOAT": "greatest of all time",

    # Work / professional
    "EOD": "end of day",
    "EOW": "end of week",
    "COB": "close of business",
    "OOO": "out of office",
    "WFH": "work from home",
    "WFO": "work from office",
    "PTO": "paid time off",
    "OOO": "out of office",
    "FYI": "for your information",
    "FYA": "for your action",
    "ETA": "estimated time of arrival",
    "ETD": "estimated time of departure",
    "KPI": "key performance indicator",
    "ROI": "return on investment",
    "ASAP": "as soon as possible",
    "EOM": "end of month",
    "TLDR": "too long did not read",
    "TL;DR": "too long did not read",

    # Technology
    "DM": "direct message",
    "PM": "private message",
    "IRL": "in real life",
    "AI": "artificial intelligence",
    "ML": "machine learning",
    "API": "application programming interface",
    "UI": "user interface",
    "UX": "user experience",
    "BTW": "by the way",
    "FAQ": "frequently asked questions",

    # Gaming
    "GG": "good game",
    "GGWP": "good game well played",
    "GLHF": "good luck have fun",
    "AFK": "away from keyboard",
    "BRB": "be right back",
    "OP": "overpowered",
    "NERF": "reduce power",
    "BUFF": "increase power",
    "NPC": "non-player character",
    "XP": "experience points",
    "PVP": "player versus player",
    "PVE": "player versus environment",
    "DLC": "downloadable content",
    "HP": "health points",
    "MP": "magic points",
    "FPS": "frames per second",

    # Relationships
    "BF": "boyfriend",
    "GF": "girlfriend",
    "BFS": "boyfriends",
    "GFS": "girlfriends",
    "SO": "significant other",
    "MIL": "mother in law",
    "FIL": "father in law",
    "BFF": "best friend forever",

    # Money / business
    "ATM": "at the moment",
    "COD": "cash on delivery",
    "EMI": "equated monthly installment",
    "GST": "goods and services tax",
    "UPI": "unified payments interface",
    "ROI": "return on investment",
    "IPO": "initial public offering",

    # Common abbreviations
    "AKA": "also known as",
    "ASL": "age sex location",
    "DIY": "do it yourself",
    "FAQ": "frequently asked questions",
    "FYI": "for your information",
    "IMO": "in my opinion",
    "IMHO": "in my humble opinion",
    "TMI": "too much information",
    "TBA": "to be announced",
    "TBD": "to be decided",
    "RSVP": "please respond",
    "DOB": "date of birth",
    "POV": "point of view",
    "PS": "postscript",
    "AKA": "also known as",
    "EST": "estimated",
    "APP": "application",
    "INFO": "information",

    # Short forms
    "CAUSE": "because",
    "COS": "because",
    "CUZ": "because",
    "COZ": "because",
    "BCOZ": "because",
    "THO": "though",
    "THRU": "through",
    "MSG": "message",
    "PIC": "picture",
    "PICS": "pictures",
    "VID": "video",
    "VIDS": "videos",
    "DOC": "document",
    "DOCS": "documents",
    "PROB": "probably",
    "APP": "application",
    "INFO": "information",
    "CONVO": "conversation",
    "CONGRATS": "congratulations",
    "CONGRATZ": "congratulations",
    "CONGRATUL8": "congratulations",

    # Time
    "MON": "Monday",
    "TUE": "Tuesday",
    "WED": "Wednesday",
    "THU": "Thursday",
    "FRI": "Friday",
    "SAT": "Saturday",
    "SUN": "Sunday",
    "AM": "morning",
    "PM": "afternoon",

    # Miscellaneous
    "WTF": "what the heck",
    "OMG": "oh my god",
    "OML": "oh my lord",
    "JFC": "oh my god",
    "STFU": "be quiet",
    "GTFO": "get out",
    "JK": "just kidding",
    "J/K": "just kidding",
    "JIC": "just in case",
    "ICYMI": "in case you missed it",
    "LMK": "let me know",
    "LMAO": "laughing my ass off",
    "NGL": "not going to lie",
    "RN": "right now",
    "RLY": "really",
    "SRSLY": "seriously",
    "SRS": "serious",
    "OBV": "obviously",
    "DEFO": "definitely",
    "PROLLY": "probably",
    "KINDA": "kind of",
    "SORTA": "sort of",
    "GONNA": "going to",
    "WANNA": "want to",
    "GOTTA": "got to",
    "DUNNO": "do not know",
    "OUTTA": "out of",
    "C'MON": "come on",
}

In [32]:
df['overview'].shape

(80,)

In [33]:
def find_chat_words(text):
    found = []

    for word in text.split():
        if word.upper() in chat_words:
            found.append(word)

    return found

df['found_chat_words'] = df['overview'].apply(find_chat_words)

In [34]:
df[df['found_chat_words'].apply(
    lambda x: any(word.lower() == 'sun' for word in x)
)]

,adult,id,title,original_language,original_title,overview,popularity,release_date,softcore,vote_average,vote_count,corrected_overview,found_chat_words
8,False,687163,Project Hail Mary,en,Project Hail Mary,science teacher land grace wakes spaceship lig...,88.5849,2026-03-15,False,8.64,7766,science teacher land grace wakes up on a space...,[sun]


In [35]:
df['found_chat_words'].value_counts()

,count
found_chat_words,
[],76
[sun],1
[sure],1
[cap],1
[based],1


In [36]:
df.iloc[8]['overview']

'science teacher land grace wakes spaceship light years home recollection got memory returns begins uncover mission solve riddle mysterious substance causing sun die must call scientific knowledge orthodox ideas save everything earth extinction'

In [37]:
df['found_chat_words'].explode().unique()

array([nan, 'sun', 'sure', 'cap', 'based'], dtype=object)

In [38]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)
df['overview']=df['overview'].apply(chat_conversion)

In [39]:
df['overview'][8]

'science teacher land grace wakes spaceship light years home recollection got memory returns begins uncover mission solve riddle mysterious substance causing Sunday die must call scientific knowledge orthodox ideas save everything earth extinction'

Tokinizing

In [46]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize,sent_tokenize
description=df['overview'].apply(word_tokenize)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [48]:
description

,overview
0,"[abraham, asylum, destroyed, inmates, unleashe..."
1,"[altar, sang, worlds, last, airbender, learns,..."
2,"[small, woodland, creature, majestic, bird, tw..."
3,"[two, women, discover, slammed, man, also, got..."
4,"[two, mexican, officers, must, survive, final,..."
...,...
75,"[five, lovely, young, girls, hate, studying, h..."
76,"[film, tells, story, 21yearold, decides, form,..."
77,"[dictator, adenoids, henker, tries, expand, em..."
78,"[three, partisans, bound, strong, friendship, ..."


,adult,id,title,original_language,original_title,overview,popularity,release_date,softcore,vote_average,vote_count,corrected_overview,found_chat_words,overeview
0,False,1560520,Batman: Knightfall Part 1: Knightfall,en,Batman: Knightfall Part 1: Knightfall,abraham asylum destroyed inmates unleashed upo...,83.7608,2026-06-23,False,9.175,312,abraham asylum has been destroyed and all its ...,[],"[abraham, asylum, destroyed, inmates, unleashe..."
1,False,980431,Avatar Aang: The Last Airbender,en,Avatar Aang: The Last Airbender,altar sang worlds last airbender learns ancien...,90.3732,2026-07-24,False,9.142,1130,altar sang the worlds last airbender learns of...,[],"[altar, sang, worlds, last, airbender, learns,..."
2,False,1007757,Swapped,en,Swapped,small woodland creature majestic bird two natu...,40.5371,2026-05-01,False,8.869,2181,a small woodland creature and a majestic bird ...,[],"[small, woodland, creature, majestic, bird, tw..."
3,False,1632181,Accidental Partners,es,Socias por accidente,two women discover slammed man also got pregna...,18.0217,2026-03-12,False,8.850,381,two women discover they were both slammed by t...,[],"[two, women, discover, slammed, man, also, got..."
4,False,1621552,Facing El Chapo,es,La captura,two mexican officers must survive final hours ...,113.2789,2026-08-21,False,8.830,675,two mexican officers must survive the final ho...,[],"[two, mexican, officers, must, survive, final,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,False,820067,The Quintessential Quintuplets Movie,ja,映画 五等分の花嫁,five lovely young girls hate studying hire pas...,7.4879,2022-05-20,False,8.286,456,when five lovely young girls who hate studying...,[],"[five, lovely, young, girls, hate, studying, h..."
76,False,644479,Dedicated to my ex,es,Dedicada A Mi Ex,film tells story 21yearold decides form rock b...,4.3289,2019-11-01,False,8.300,514,the film tells the story of are a 21yearold wh...,[],"[film, tells, story, 21yearold, decides, form,..."
77,False,914,The Great Dictator,en,The Great Dictator,dictator adenoids henker tries expand empire p...,11.5340,1940-10-15,False,8.285,3808,dictator adenoids henker tries to expand his e...,[],"[dictator, adenoids, henker, tries, expand, em..."
78,False,42269,We All Loved Each Other So Much,it,C'eravamo tanto amati,three partisans bound strong friendship return...,5.0806,1974-12-21,False,8.284,666,three partisans bound by a strong friendship r...,[],"[three, partisans, bound, strong, friendship, ..."
